In [ ]:
!pip install -q scikit-learn pyarrow

In [ ]:
import pandas as pd
import numpy as np
from sklearn.ensemble import IsolationForest
from sklearn.metrics import precision_score, recall_score, f1_score

In [ ]:
INPUT_DIR = "<link-to-the-4th-notebook>"
holdout_preds = pd.read_parquet(f"{INPUT_DIR}/final_holdout_predictions.parquet")
holdout_preds["date"] = pd.to_datetime(holdout_preds["date"])
holdout_preds["residual"] = holdout_preds["sales"] - holdout_preds["forecast"]

In [ ]:
print(holdout_preds.shape)
print(holdout_preds.head())

In [ ]:
def control_limit_flags(df, residual_col="residual", std_reference_col=None, k=2.5):
    df = df.copy()
    if std_reference_col is None:
        std_reference = df.groupby(["store_nbr", "family"])[residual_col].transform("std")
    else:
        std_reference = df[std_reference_col]
    df["control_limit_flag"] = (df[residual_col].abs() > k * std_reference).astype(int)
    return df

In [ ]:
holdout_preds = control_limit_flags(holdout_preds)
print(f"Control-limit flagged anomalies: {holdout_preds['control_limit_flag'].sum()} / {len(holdout_preds)}")

In [ ]:
def build_scaled_iso_features(df, residual_col, sales_col, forecast_col):
    features = pd.DataFrame(index=df.index)
    for col in [residual_col, sales_col, forecast_col]:
        group_mean = df.groupby(["store_nbr", "family"])[col].transform("mean")
        group_std = df.groupby(["store_nbr", "family"])[col].transform("std").replace(0, np.nan)
        features[col] = (df[col] - group_mean) / group_std
    return features.fillna(0)

In [ ]:
iso_features = build_scaled_iso_features(holdout_preds, "residual", "sales", "forecast")
iso_model = IsolationForest(contamination=0.05, random_state=42)
holdout_preds["isoforest_flag"] = (iso_model.fit_predict(iso_features) == -1).astype(int)
print(f"IsolationForest flagged anomalies: {holdout_preds['isoforest_flag'].sum()} / {len(holdout_preds)}")

In [ ]:
def inject_synthetic_anomalies(df, n_anomalies=50, spike_multiplier_range=(2.5, 5.0),
                                 drop_fraction_range=(0.5, 0.9), seed=42):
    """
    Spikes: sales_injected = baseline * (1 + multiplier) -> graded 3.5x-6x upward spike.
    Drops:  sales_injected = baseline * (1 - drop_fraction) -> graded 50-90% reduction,
            never a hard collapse to zero.
    """
    rng = np.random.default_rng(seed)
    df = df.copy().reset_index(drop=True)
    df["is_synthetic_anomaly"] = 0
    df["sales_injected"] = df["sales"].astype(float)

    # Per-series mean, not the global mean across all store/family combos - a DAIRY series
    # and a GROCERY I series can differ by an order of magnitude, and a spike/drop sized off
    # the wrong scale produces an unrealistic synthetic anomaly (trivially easy or impossible
    # to detect depending on which series it lands in).
    series_mean = df.groupby(["store_nbr", "family"])["sales"].transform("mean")

    anomaly_idx = rng.choice(df.index, size=min(n_anomalies, len(df)), replace=False)
    for idx in anomaly_idx:
        direction = rng.choice(["spike", "drop"])
        row_sales = df.loc[idx, "sales"]
        row_series_mean = series_mean.loc[idx]
        baseline = row_sales if row_sales > 0 else (row_series_mean if row_series_mean > 0 else df["sales"].mean())
        if direction == "spike":
            multiplier = rng.uniform(*spike_multiplier_range)
            injected_value = baseline * (1 + multiplier)
        else:
            drop_frac = rng.uniform(*drop_fraction_range)
            injected_value = baseline * (1 - drop_frac)
        df.loc[idx, "sales_injected"] = max(0, injected_value)
        df.loc[idx, "is_synthetic_anomaly"] = 1

    df["residual_injected"] = df["sales_injected"] - df["forecast"]
    return df

In [ ]:
eval_df = inject_synthetic_anomalies(holdout_preds)

In [ ]:
clean_std_map = (
    holdout_preds.groupby(["store_nbr", "family"])["residual"].std()
    .rename("clean_residual_std")
)
eval_df = eval_df.merge(clean_std_map, on=["store_nbr", "family"], how="left")
eval_df = control_limit_flags(eval_df, residual_col="residual_injected", std_reference_col="clean_residual_std")
eval_df = eval_df.rename(columns={"control_limit_flag": "control_limit_flag_injected"})

In [ ]:
# Bug fix: this used to normalize with mean/std computed on eval_df itself, i.e. on the
# injected column - so the 50 synthetic spikes/drops were included in the very group
# stats used to detect them. With only ~15 holdout rows per store/family series, one or
# two injected points meaningfully shift that group's own mean/std, unlike control limits
# above, which explicitly reference clean_std_map. That gave IsolationForest a contaminated
# baseline while control limits got a clean one - not a fair comparison between methods.
# Fix: build the scaling stats from the clean (pre-injection) holdout_preds, then apply
# that same clean scaling to the injected values, mirroring how control_limit_flags does it.
def build_scaled_iso_features_from_clean_reference(df, clean_reference_df, value_cols):
    """Scales each column in `df` using mean/std computed from `clean_reference_df`
    (grouped by store_nbr/family), not from `df` itself."""
    keys = df[["store_nbr", "family"]].reset_index(drop=True)
    features = pd.DataFrame(index=df.index)
    for injected_col, clean_col in value_cols.items():
        clean_mean = clean_reference_df.groupby(["store_nbr", "family"])[clean_col].transform("mean")
        clean_std = clean_reference_df.groupby(["store_nbr", "family"])[clean_col].transform("std")
        clean_stats = clean_reference_df[["store_nbr", "family"]].copy()
        clean_stats["mean"] = clean_mean
        clean_stats["std"] = clean_std
        clean_stats = clean_stats.drop_duplicates(subset=["store_nbr", "family"])

        merged = keys.merge(clean_stats, on=["store_nbr", "family"], how="left")
        group_mean = merged["mean"].values
        group_std = merged["std"].replace(0, np.nan).values
        features[injected_col] = (df[injected_col].values - group_mean) / group_std
    return features.fillna(0)

In [ ]:
iso_features_injected = build_scaled_iso_features_from_clean_reference(
    eval_df, holdout_preds,
    value_cols={"residual_injected": "residual", "sales_injected": "sales", "forecast": "forecast"},
)
iso_model_injected = IsolationForest(contamination=0.05, random_state=42)
eval_df["isoforest_flag_injected"] = (
    iso_model_injected.fit_predict(iso_features_injected) == -1
).astype(int)

In [ ]:
metrics = {}
for method in ["control_limit_flag_injected", "isoforest_flag_injected"]:
    y_true = eval_df["is_synthetic_anomaly"]
    y_pred = eval_df[method]
    metrics[method] = {
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
    }

In [ ]:
metrics_df = pd.DataFrame(metrics).T
print(metrics_df)

In [ ]:
holdout_preds.to_parquet("/kaggle/working/anomaly_results.parquet", index=False)
metrics_df.to_csv("/kaggle/working/anomaly_eval_metrics.csv")
print("Saved: anomaly_results.parquet, anomaly_eval_metrics.csv")